# Stage 6.5: 亚群 Subset 重分析

面向 **非计算机专业 PI/学生**（ADR-0009）。

当 PI 在 stage6 全局注释后希望对某类细胞（如所有 T 细胞 / 上皮细胞 /
SPEM 谱系）做更精细的亚群分析时，本 notebook 执行以下流程：

1. **子集抽取**：按 `SUBSET_FILTER` 表达式筛选（如 `cell_type_final_v1.isin(['CD4 T', 'CD8 T', 'Treg'])`）
2. **Stage 3 重跑**：对子集重新选 HVG（全局 HVG 对亚群未必最优）
3. **Stage 4 重跑**：对子集重新做嵌入（子集的批次效应/生物变异结构不同）
4. **Stage 5 重跑**：对子集重新聚类（子集通常需要更细的分辨率）
5. **Stage 6 重跑**：对子集重新标注（精细细胞亚型，如 T_cell → CD4_Tcm/CD8_Tem）
6. **标签回流**：将精细标签写回主图谱的新列 `cell_type_final_subset_v1`——
   **子集内的细胞**获得精细标签，**子集外的细胞保留 NaN**。
   原 `cell_type_final_v1` 列**不覆盖**——保留层级粒度（粗在 `_final_v1`，细在 `_final_subset_v1`）。

**为什么需要亚群重分析？**
全局分析的目标是区分大类（上皮 vs 免疫 vs 间质），参数为这个目标校准。
但同一大类内部的异质性（如 T 细胞的 CD4/CD8 亚群、上皮的 pit/neck/SPEM 梯度）
需要重新选 HVG + 重新聚类才能看清——这是 scRNA-seq 分析的标准实践，
不是"重跑浪费算力"。

**溯源机制**：`adata_sub.uns["subset_of"]` 记录来源 h5ad，
`adata_sub.uns["subset_filter"]` 记录筛选表达式，
任何下游分析都能追溯这个子集是如何产生的（见 SPEC 196）。

**实现纪律（ADR-0003/0009）**：直接调 scanpy 原生 API，
无 plugin/registry/class。每个 stage 是独立的 cell block，
非 CS 学生按顺序读下来能看懂每一步在做什么。

## 如何回跑（迭代调参机制）

### 管线位置
- **上游**：Stage 6（注释），读 `stage6_annotated_v*.h5ad`
- **下游**：Stage 7（下游分析），产出两个文件：
  - `stage6_5_subset_v*.h5ad`（子集独立 h5ad）
  - `stage6_annotated_v*.h5ad`（主 adata 回流更新版，含 `cell_type_final_subset_v1` 列）

### 为什么要迭代回跑？
亚群重分析的质量取决于上游注释的准确性 + 子集筛选的合理性 + 子集内参数的选择。
当以下任一情况发生时需要重跑：
- PI 发现子集注释不够精细（如 T 细胞应拆成 CD4/CD8/Treg 但未拆）
- PI 调整了全局的 `cell_type_final_v1` 标签（上游 stage 6 重跑后）
- PI 想分析另一类细胞（改 `SUBSET_FILTER`）
- 子集 Leiden 聚类结果不理想（改 `RESOLUTIONS` 或 `N_PCS`）
- 标记物 CSV 更新（改 `MARKER_CSV`）

PI 的原始要求（来自项目构思）：
> "注释这一步，也可能在注释的过程中发现前面高可变基因的选择、embedding 的构建，
> 还有分群的参数等等需要调整，应可以随时调回去重跑一些流程，需要建立这种循环不断迭代的机制。"

### 如何回跑（三步操作）
1. **改 `UPSTREAM_PATH`**——指向要复用的上游文件版本
   （例如 `nancang_stage6_annotated_v2.h5ad`）
2. **改 `OUTPUT_PATH`**——bump 子集版本号 `_v1` → `_v2`
   （例如 `nancang_stage6_5_subset_v2.h5ad`）
   同时改 `MAIN_OUTPUT_PATH`——bump 主 adata 版本号
   （例如 `nancang_stage6_annotated_v3.h5ad`）
3. **调整参数**（在下方 `# === PARAMS ===` 区域改 `SUBSET_FILTER`、`N_PCS`、
   `RESOLUTIONS` 或标记物参数）
   → 重跑本 notebook（Cell → Run All）

### 版本约定
- **`_v1` / `_v2` / ...**：每次调参重跑 bump 一位版本号。
  旧版 `.h5ad` 文件**不覆盖不删除**，保留在 `results/` 目录供追溯对比。
- **`experimental`**：刚跑出、尚未经 PI 审查确认的版本（默认值）。
- **`promoted`**：PI 审查后认为子集分析质量可接受、可传给下游使用的正式版本。

### 两个输出对象的溯源策略（重要）
本 notebook 产出两个 h5ad，但**区别对待**：
- **`adata_sub`（子集对象）**：全新对象，独立顶层追踪字段
  （`stage="stage6_5_subset"`、`version="v1"`、`upstream=[UPSTREAM_PATH]`、`status="experimental"`）。
  保留其 `stage6_5_v1` 嵌套 dict 作为细节记录。
- **`main_adata`（主对象回流更新）**：本身是 stage 6 的产物，**不覆盖**
  其顶层 `stage` / `version` / `upstream` 字段——那会篡改 stage 6 的溯源链。
  `main_adata` 仅写入 `stage6_5_reflow_v1` 嵌套 dict 记录本次回流操作。
  主对象的追踪字段由 stage 6 负责维护。

### 追溯链
如需查询"stage 6.5 有哪些版本？"或"这个子集依赖哪个 stage 6 版本？"，
可在 Python 中检查子集 h5ad 的 `adata.uns["stage"]` / `adata.uns["upstream"]`。


In [ ]:
# === PARAMS ===
# UPSTREAM_PATH       -- stage6_annotated 输出 h5ad（必须有 cell_type_final_v1）
# SUBSET_FILTER       -- 筛选表达式（Python 表达式，作用在 adata.obs 上）
# OUTPUT_PATH         -- 子集 h5ad 版本化输出（stage 6.5 产物）。
#                        版本号 _v1 与 adata_sub.uns["version"] 保持一致。
#                        如需回跑：bump 版本号 _v1->v2，旧版不覆盖。
# MAIN_OUTPUT_PATH    -- 主 adata 更新后的输出（含 cell_type_final_subset_v1 列）
# N_TOP_GENES         -- HVG 数量
# N_PCS               -- PCA 主成分数
# RESOLUTIONS         -- Leiden 多分辨率列表
# MARKER_CSV          -- 标记物知识库 CSV
# LLM_MODELS          -- mLLMCelltype 模型列表（key 守卫）
# CONSENSUS_MODEL     -- mLLMCelltype 共识讨论模型
# BASE_URLS           -- provider base URLs
# RANDOM_SEED         -- 随机种子（全流程一致）

UPSTREAM_PATH  = "results/nancang_stage6_annotated_v1.h5ad"
SUBSET_FILTER  = "cell_type_final_v1.isin(['CD4 T', 'CD8 T', 'Treg', 'NK cell', 'B cell'])"
OUTPUT_PATH    = "results/nancang_stage6_5_subset_v1.h5ad"

# 主 adata 更新输出（版本号随着每次 subset 回流递增）
MAIN_OUTPUT_PATH = "results/nancang_stage6_annotated_v2.h5ad"

# ---- 分析参数 ----
N_TOP_GENES = 3000
N_PCS       = 50
RESOLUTIONS = [0.4, 0.6, 0.8, 1.0, 1.2, 1.6]
MARKER_CSV  = "references/markers/gastric_epithelial.csv"

# ---- mLLMCelltype 多模型共识 ----
LLM_MODELS = [
    "openai/gpt-4.1-nano",
    "anthropic/claude-haiku-4-5",
    "deepseek/deepseek-chat",
]
CONSENSUS_MODEL = "anthropic/claude-sonnet-4-6"

BASE_URLS = {
    "openai":    "",
    "anthropic": "",
    "deepseek":  "https://api.deepseek.com",
    "qwen":      "",
    "gemini":    "",
}

RANDOM_SEED = 42

# 每运行一次 stage 6.5 递增 N
# 如 T cells 做了一次，上皮又做了一次 → N 分别 = 1, 2

In [ ]:
# 确保框架 src/ 在 sys.path 并切换到项目根目录。
import sys, os, gc, datetime, warnings
_root = os.getcwd()
if not os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
    _root = os.path.abspath(os.path.join(_root, ".."))
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures/stage6_5", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")

# 导入（scanpy 原生 + 框架函数）
import scanpy as sc
import scvi
import scipy.sparse as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scrna_integration import load_markers

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
warnings.filterwarnings("ignore", category=FutureWarning)

# 固定随机种子（全流程一致，保证可复现）
np.random.seed(RANDOM_SEED)

# 加载上游 adata
print(f"加载上游: {UPSTREAM_PATH}")
adata = sc.read_h5ad(UPSTREAM_PATH)
print(f"已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")
print(f"X dtype: {adata.X.dtype}  |  sparse: {sp.issparse(adata.X)}")

## 子集抽取

按 `SUBSET_FILTER` 表达式筛选细胞，`copy()` 保证子集独立于主 adata。
在子集 `uns` 中记录 `subset_of` 和 `subset_filter` 以供溯源。

**为什么用 `.copy()` 而非 view？**
后面的 HVG 重选、归一化、嵌入都需要独立的内存空间。
view 会意外修改主 adata 的数据，copy 隔离这个风险。

**为什么先确认 `cell_type_final_v1` 存在？**
subset 筛选依赖于这个列——如果该列不存在（如 PI 还没拍板），
整条分析链路无意义。提前报错比后面发现好。

In [ ]:
# 确认 cell_type_final_v1 存在（subset 筛选依赖它）。
if "cell_type_final_v1" not in adata.obs.columns:
    raise KeyError(
        "obs 中没有 'cell_type_final_v1' 列。"
        "请先运行 stage6_annotated.ipynb 并完成 PI 拍板（pi_decisions）。"
    )

# 执行子集筛选——为什么用 eval？让 PI 在 PARAMS 中写人类可读的表达式，
# 而非在代码里硬编码标签列表。
print(f"筛选表达式: {SUBSET_FILTER}")
_mask = adata.obs.eval(SUBSET_FILTER)
print(f"筛选前: {adata.n_obs:,} 细胞")
print(f"筛选后: {_mask.sum():,} 细胞 ({_mask.sum()/adata.n_obs*100:.1f}%)")

if _mask.sum() < 50:
    raise ValueError(
        f"子集仅 {_mask.sum()} 细胞，不足以做有意义的亚群分析。"
        "请检查 SUBSET_FILTER。"
    )

# copy() 保证独立内存空间
adata_sub = adata[_mask].copy()
print(f"\n子集 adata: {adata_sub.n_obs:,} 细胞 x {adata_sub.n_vars:,} 基因")

# ---- 溯源元数据 ----
# 为什么记录这些？任何下游分析或合作者拿到这个 h5ad 后，
# 通过 uns["subset_of"] 和 uns["subset_filter"] 就能追溯来源——
# 不需要翻 notebook 找参数。
adata_sub.uns["subset_of"] = UPSTREAM_PATH
adata_sub.uns["subset_filter"] = SUBSET_FILTER
adata_sub.uns["subset_n_cells_before"] = adata.n_obs
adata_sub.uns["subset_n_cells_after"] = adata_sub.n_obs
print(f"溯源信息已记录: subset_of={UPSTREAM_PATH}")
print(f"                   subset_filter={SUBSET_FILTER}")

# 释放主 adata（后续不需要它，直到回流步骤）
del adata
gc.collect()
print("主 adata 已释放")

## Stage 3: 归一化 + 高变基因重选

**为什么子集要重选 HVG？**
全局 HVG 选的是能区分所有大类（上皮 vs 免疫 vs 间质）的基因。
但在 T 细胞子集中，这些基因大部分不表达（如上皮特异基因 MUC5AC），
真正区分 CD4/CD8/Treg 的基因（如 CD4/CD8A/FOXP3）在全局 HVG 中
可能因为跨大类差异不够大而被过滤掉。
因此子集分析的第一步永远是 **对子集重新选 HVG**。

**为什么先存 counts 层？**
归一化会覆盖 `adata.X`，但 stage 6 的基因集评分等下游可能需要 raw counts。
存到 `layers["counts"]` 是 scanpy 标准实践，学生应该学会这个习惯。

In [ ]:
# Stage 3: 归一化 + log + HVG（在子集上重跑）。
print("=== Stage 3: normalize + HVG re-selection on subset ===")

# 保留 raw counts（归一化前）
# 为什么存 layers["counts"]？这是 scanpy 社区约定——
# 任何下游分析需要 raw counts 时从这里取，不用回头找 stage2 输出。
adata_sub.layers["counts"] = adata_sub.X.copy()

# 归一化到 10,000 counts per cell
# 为什么 target_sum=1e4？这是 scRNA-seq 的标准归一化目标——
# 与 10x Genomics 的默认值、Cell Ranger 的输出口径一致。
sc.pp.normalize_total(adata_sub, target_sum=1e4)
sc.pp.log1p(adata_sub)

# HVG 重选——用 seurat_v3 flavor
# 为什么 flavor="seurat_v3"？seurat_v3 基于方差稳定化变换选 HVG，
# 对比默认的 seurat（基于 dispersion），在子集分析中对稀有亚群的标记
# 基因检出率更高。见 ADR-0009 注释中文化时对 flavor 选择的讨论。
sc.pp.highly_variable_genes(
    adata_sub, n_top_genes=N_TOP_GENES, flavor="seurat_v3",
)
# 也可以额外指定 batch_key 避免批次驱动 HVG 选择：
# sc.pp.highly_variable_genes(
#     adata_sub, n_top_genes=N_TOP_GENES, flavor="seurat_v3",
#     batch_key="source_dataset",
# )

n_hvg = adata_sub.var["highly_variable"].sum()
print(f"\nHVG 选择: {n_hvg}/{adata_sub.n_vars} 基因标记为 highly_variable")
print(f"  前 10 个 HVG: {list(adata_sub.var_names[adata_sub.var['highly_variable']][:10])}")

# 可视化检查
sc.pl.highly_variable_genes(adata_sub, show=False)
plt.savefig("results/figures/stage6_5_hvg.png", dpi=120, bbox_inches="tight")
plt.close()
print("  HVG 图已保存: results/figures/stage6_5_hvg.png")

# 内存纪律：归一化不改变 sparse 性质，但确认 dtype
adata_sub.X = adata_sub.X.astype(np.float32)
assert sp.issparse(adata_sub.X) and adata_sub.X.dtype == np.float32
print("  内存自检: X sparse CSR float32 OK")

# 记录运行元数据
adata_sub.uns["normalize_v1"] = {
    "target_sum": 1e4,
    "hvg_flavor": "seurat_v3",
    "n_top_genes": N_TOP_GENES,
    "n_hvg": int(n_hvg),
    "subset": True,  # 标记这是子集重跑
    "timestamp": datetime.datetime.now().isoformat(),
}

## Stage 4: 多方法嵌入（在子集上）

与全局 stage4 相同的方法阵容，但在子集上独立运行。

**为什么子集需要重新嵌入？**
全局 PCA 的 loadings 由所有细胞的方差结构决定——在上皮细胞主导的
全局数据中，PC1-3 大概率是上皮 vs 免疫的差异。T 细胞子集的内部
变异（CD4 vs CD8、naive vs memory）在这些 PC 上可能是噪声。
在子集上重新跑 PCA+整合，嵌入空间才能真正反映子集内部的生物学变异。

**批次整合**：子集通常跨多个 source_dataset（病种/平台），
仍需要 Harmony/scVI 去批次。使用与全局相同的 `batch_key="source_dataset"`。

In [ ]:
# Stage 4: PCA + Harmony + scVI（在子集上独立运行）。
print("=== Stage 4: embedding on subset ===")

# ---- PCA ----
sc.tl.pca(adata_sub, n_comps=N_PCS, use_highly_variable=True, svd_solver="arpack")
print(f"PCA 完成: {N_PCS} PCs")

# ---- Harmony 去批次 ----
# 为什么 batch_key="source_dataset"？不同数据集可能存在不同的技术/平台
# 批效应。Harmony 在 PCA 空间做 soft-clustering 对齐，
# 对 scRNA-seq 的稀疏数据比 MNN/CCA 更快且不易过校正。
sc.external.pp.harmony_integrate(
    adata_sub, key="source_dataset",
)
print("Harmony 完成: obsm['X_pca_harmony']")

# ---- scVI（深度生成模型）----
# scVI 假设计数数据服从零膨胀负二项分布，从 raw counts 学习隐变量。
# 为什么用 counts 层而非 normalized X？scVI 内部有自己的归一化机制，
# 直接用 counts 避免丢失分布信息。
scvi.model.SCVI.setup_anndata(
    adata_sub, batch_key="source_dataset", layer="counts",
)
_model = scvi.model.SCVI(adata_sub, n_latent=30, n_layers=2)
# max_epochs 可根据子集大小调整——子集小（<5000 cells）用较少 epoch
_max_epochs = 200 if adata_sub.n_obs > 5000 else 100
_model.train(max_epochs=_max_epochs, early_stopping=True)
adata_sub.obsm["X_scVI"] = _model.get_latent_representation()
print(f"scVI 完成: obsm['X_scVI'] shape={adata_sub.obsm['X_scVI'].shape}")

# 记录元数据
adata_sub.uns["embedding_v1"] = {
    "methods": ["pca", "harmony", "scvi"],
    "n_pcs": N_PCS,
    "n_latent": 30,
    "batch_key": "source_dataset",
    "subset": True,
    "timestamp": datetime.datetime.now().isoformat(),
}
print("  嵌入元数据已记录")

## Stage 5: 多分辨率聚类（在子集上）

用 Harmony 嵌入构建 kNN 图，然后做多分辨率 Leiden 聚类。
**为什么用 Harmony embedding 建图？** Harmony 已去除已知批次效应，
在此基础上建图能减少"同细胞类型因技术差异被拆成不同簇"的问题。

**为什么在子集上用更细的分辨率（1.0–1.6）？**
子集中的生物学差异比全局更 subtle——CD4 Tcm vs CD4 Tem 之间的
标记基因差异远小于 T cell vs epithelial 之间的差异。
更细的分辨率让 Leiden 能捕捉这些微妙结构。

PI 在多个分辨率中选最合理的，方法与全局 stage5 相同。

In [ ]:
# Stage 5: 多分辨率 Leiden 聚类（在子集上）。
print("=== Stage 5: clustering on subset ===")

# 用 Harmony 嵌入构建 kNN 图
# 为什么 use_rep="X_pca_harmony"？去批次后再建邻居关系，
# 避免"相同生物学但因批次差异被归为不同簇"。
_use_rep = "X_pca_harmony"
if _use_rep not in adata_sub.obsm:
    # fallback: 如果 Harmony 没跑，用 PCA
    _use_rep = "X_pca"
    print(f"  Harmony 不可用，改用 {_use_rep}")

sc.pp.neighbors(adata_sub, use_rep=_use_rep, n_neighbors=15)

# 计算 UMAP（供后续可视化）
sc.tl.umap(adata_sub)
print(f"UMAP 完成: obsm['X_umap']")

# 多分辨率 Leiden clustering
for res in RESOLUTIONS:
    key = f"leiden_res_{res}"
    sc.tl.leiden(adata_sub, resolution=res, key_added=key)
    n_clusters = adata_sub.obs[key].nunique()
    print(f"  leiden_res_{res}: {n_clusters} 簇")

print(f"\n多分辨率聚类完成。{len(RESOLUTIONS)} 个分辨率已写入 obs。")

# 可视化各分辨率
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
for ax, res in zip(axes, RESOLUTIONS):
    sc.pl.umap(
        adata_sub, color=f"leiden_res_{res}", ax=ax,
        title=f"Leiden res={res}", legend_loc="right margin",
        show=False,
    )
for ax in axes[len(RESOLUTIONS):]:
    ax.set_visible(False)
fig.tight_layout()
fig.savefig("results/figures/stage6_5_leiden_sweep.png", dpi=120, bbox_inches="tight")
plt.close(fig)
print("  Leiden sweep 图已保存: results/figures/stage6_5_leiden_sweep.png")

# 记录元数据
adata_sub.uns["clustering_v1"] = {
    "use_rep": _use_rep,
    "resolutions": RESOLUTIONS,
    "n_neighbors": 15,
    "subset": True,
    "timestamp": datetime.datetime.now().isoformat(),
}

### PI 选择子集聚类分辨率

多分辨率中 PI 选择一个最合理的分辨率作为本子集的标签列。
通常的原则是：簇数不能太多（过度分裂）也不能太少（欠聚类），
要能在 UMAP 上看到清晰的群体分离。

In [ ]:
# === PI 选择子集聚类分辨率 ===
# PI 查看上方的 Leiden sweep UMAP 图后，选择一个分辨率。
# 为什么让 PI 选？自动选分辨率的算法（如 silhouette 最大值）
# 对子集分析的 subtle 结构不够敏感——PI 的领域知识在这里不可替代。
LEIDEN_COL = "leiden_res_0.6"  # PI 从上方图中选一个最合理的分辨率

if LEIDEN_COL not in adata_sub.obs.columns:
    print(f"⚠ 警告: '{LEIDEN_COL}' 不在 obs 中。可用列: "
          f"{[c for c in adata_sub.obs.columns if c.startswith('leiden_')]}")
else:
    n_clust = adata_sub.obs[LEIDEN_COL].nunique()
    print(f"选用聚类列: {LEIDEN_COL} ({n_clust} 簇)")
    # 确保是 categorical
    if not pd.api.types.is_categorical_dtype(adata_sub.obs[LEIDEN_COL]):
        adata_sub.obs[LEIDEN_COL] = adata_sub.obs[LEIDEN_COL].astype("category")

## Stage 6: 子集重标注（精细细胞亚型）

与全局 stage6 相同的多方法标注流程，但目标改为精细亚型：
- 全局 stage6 标注的是：T_cell, B_cell, pit_cell, SPEM...
- 子集 stage6 标注的是：CD4_Tcm, CD8_Tem, Treg, MAIT...（在 T 细胞子集内）

**为什么用多方法？**
与全局 stage6 相同的理由——单方法容易出错，多方法交叉比对才能
找到置信度高的标签。但这里调用 mLLMCelltype 时，tissue prompt
应调整为子集对应的 context（如 "human peripheral blood T cells"
而非 "human stomach"）。

**注意**：本 cell 的 LLM 调用同样受 key 守卫——无 key 时优雅跳过。

In [ ]:
# Stage 6: 子集重标注（精细细胞亚型）。
print("=== Stage 6: re-annotation on subset (finer cell types) ===")

# ---- 方法 1: 标记物 dotplot（PI 手动标注）----
_markers = load_markers(MARKER_CSV)
_all_marker_genes = sorted(set(g for glist in _markers.values() for g in glist))
_avail = [g for g in _all_marker_genes if g in adata_sub.var_names]
_avail = _avail[:30]  # 子集分析减少展示基因数避免图过大

if _avail and LEIDEN_COL in adata_sub.obs.columns:
    sc.pl.dotplot(
        adata_sub, var_names=_avail, groupby=LEIDEN_COL,
        dendrogram=True, standard_scale="var",
        title=f"Canonical marker dotplot (subset, {LEIDEN_COL})",
        show=False,
    )
    plt.savefig("results/figures/stage6_5_dotplot.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("  标记物 dotplot 已保存: results/figures/stage6_5_dotplot.png")

# ---- 方法 2: 基因集评分（scanpy score_genes）----
_score_cols = []
for ct, gene_list in _markers.items():
    _present = [g for g in gene_list if g in adata_sub.var_names]
    if len(_present) < 2:
        continue
    col = f"score_{ct}"
    sc.tl.score_genes(
        adata_sub, gene_list=_present, score_name=col,
        ctrl_size=max(1, min(len(_present), 50)),
    )
    _score_cols.append(col)
print(f"  基因集评分: {len(_score_cols)} 个评分列 -> obs")

# ---- 方法 3: mLLMCelltype 多模型共识（key 守卫）----
from dotenv import load_dotenv
load_dotenv()

_llm_providers = ["OPENAI", "ANTHROPIC", "DEEPSEEK", "QWEN", "GEMINI"]
_has_key = any(
    os.getenv(f"{p}_API_KEY") not in (None, "")
    for p in _llm_providers
)

if _has_key:
    import mllmcelltype as mct

    sc.tl.rank_genes_groups(
        adata_sub, groupby=LEIDEN_COL, method="wilcoxon",
        n_genes=30, key_added="rank_genes_stage6_5",
    )

    if not pd.api.types.is_categorical_dtype(adata_sub.obs[LEIDEN_COL]):
        adata_sub.obs[LEIDEN_COL] = adata_sub.obs[LEIDEN_COL].astype("category")
    _cluster_ids = adata_sub.obs[LEIDEN_COL].cat.categories.tolist()
    _marker_dict = {}
    for _cid in _cluster_ids:
        _df = sc.get.rank_genes_groups_df(
            adata_sub, group=_cid, key="rank_genes_stage6_5"
        )
        _marker_dict[_cid] = _df["names"].head(20).tolist()

    print(f"  启动 mLLMCelltype 共识注释（subset，{len(_marker_dict)} 个簇）...")
    _result = mct.interactive_consensus_annotation(
        marker_genes=_marker_dict,
        species="human",
        tissue="stomach",  # PI 可据子集调整（如 immune cells 改为 "blood/immune"）
        models=LLM_MODELS,
        consensus_model=CONSENSUS_MODEL,
        consensus_threshold=0.7,
        entropy_threshold=1.0,
        max_discussion_rounds=3,
        base_urls=BASE_URLS,
    )

    _consensus_map = _result.get("consensus", {})
    adata_sub.obs["cell_type_llm_subset_v1"] = (
        adata_sub.obs[LEIDEN_COL].astype(str).map(_consensus_map)
    )
    _controversial = _result.get("controversial_clusters", [])
    print(f"  LLM 共识: {len(_consensus_map)} 个簇已标注")
    print(f"  有争议簇: {_controversial}")
    adata_sub.uns["cell_type_llm_subset_v1_meta"] = {
        "models": LLM_MODELS,
        "consensus_model": CONSENSUS_MODEL,
        "controversial_clusters": _controversial,
        "timestamp": datetime.datetime.now().isoformat(),
    }
else:
    print("  " + "=" * 60)
    print("  mLLMCelltype 已跳过——未检测到 LLM API key。")
    print("  请在 .env 中配置后重新运行本 cell。")
    print("  " + "=" * 60)
    adata_sub.obs["cell_type_llm_subset_v1"] = np.nan
    adata_sub.obs["cell_type_llm_subset_v1"] = (
        adata_sub.obs["cell_type_llm_subset_v1"].astype("category")
    )

### PI 拍板子集精细标签

PI 查看上方 dotplot + LLM 共识结果后，为子集的每个簇填入精细细胞类型标签。
这些标签将回流到主 adata 的 `cell_type_final_subset_v1` 列。

**命名建议**：精细标签应比全局标签更具体——
如全局标签 `T_cell` → 精细标签 `CD4_Tcm`、`CD8_Tem`、`Treg` 等。

In [ ]:
# === PI 拍板子集精细标签 ===
# PI 根据 dotplot + LLM 共识 + 基因集评分，为子集每个簇填入精细标签。
# 为什么精细标签与全局标签分开列？这是 SPEC 835 的层级粒度设计：
# cell_type_final_v1 保持大类标签（跨病种可比），
# cell_type_final_subset_v1 提供精细亚型（仅在子集内有意义）。
pi_subset_decisions = {
    # "0": "CD4_Tcm",
    # "1": "CD8_Tem",
    # "2": "Treg",
    # ... PI 逐簇填入
}

if pi_subset_decisions and LEIDEN_COL in adata_sub.obs.columns:
    adata_sub.obs["cell_type_final_subset_v1"] = (
        adata_sub.obs[LEIDEN_COL].astype(str).map(pi_subset_decisions)
    )
    n_assigned = adata_sub.obs["cell_type_final_subset_v1"].notna().sum()
    print(f"子集精细标注: {n_assigned}/{adata_sub.n_obs} 细胞已标注")
    print(f"  标签种类: {adata_sub.obs['cell_type_final_subset_v1'].nunique()}")
else:
    adata_sub.obs["cell_type_final_subset_v1"] = np.nan
    adata_sub.obs["cell_type_final_subset_v1"] = (
        adata_sub.obs["cell_type_final_subset_v1"].astype("category")
    )
    print("PI 暂未填写子集精细标签，已预建 cell_type_final_subset_v1 空列")

## 标签回流：将子集精细标签写回主 adata

这是 stage 6.5 最关键的一步。子集的精细标签只有回流到主 adata 的
`cell_type_final_subset_v1` 列后，下游 stage7 模块才能同时利用两个层级：

- `cell_type_final_v1`：大类标签（所有细胞都有，适合跨病种比较）
- `cell_type_final_subset_v1`：精细标签（仅子集细胞有值，其余 NaN）

**为什么不动 `cell_type_final_v1`？**
这是 SPEC 835 的硬约束——两个粒度的标签共存而不互相污染。
如果一个分析只需要大类，用 `cell_type_final_v1`；
如果需要在 T 细胞内部做精细 DEG，可以用 `cell_type_final_subset_v1` 过滤 NaN。

**为什么主 adata 回写版本号递增（v1→v2）？**
回流修改了主 adata 的 obs 列——这是一个新的数据状态，
应按版本化约定创建新版本而非覆盖原版。下游 notebook 指定
合适的版本号即可。

In [ ]:
# 标签回流：将子集精细标签写回主 adata。
print("=== 标签回流 ===")

# 重新加载主 adata（上游 path——而非更新后的版本，避免循环）
print(f"加载主 adata: {UPSTREAM_PATH}")
main_adata = sc.read_h5ad(UPSTREAM_PATH)

# 回流前快照：保存 cell_type_final_v1 用于回流后真实验证
# 为什么先存快照？只有 before/after 比对才能真正确认未修改——
# 不能靠读取一个从未写入的 uns key 冒充验证。
_check = main_adata.obs["cell_type_final_v1"].copy()

# 初始化 cell_type_final_subset_v1 列——全部 NaN
main_adata.obs["cell_type_final_subset_v1"] = np.nan

# 对齐：子集细胞 → 主 adata 中的对应细胞
# 为什么用 index 对齐？保证一一对应——子集是从主 adata copy 出来的，
# 原始 index 与主 adata 相同。
_sub_labels = adata_sub.obs["cell_type_final_subset_v1"]
_common_idx = main_adata.obs_names.intersection(adata_sub.obs_names)
print(f"可回流细胞: {len(_common_idx):,} / {adata_sub.n_obs:,} 子集细胞")

# 将精细标签写入主 adata 对应细胞
# 为什么不用 .values？pandas 自动按 index 对齐赋值——
# .values 剥掉 index 后变成裸 numpy array，依赖顺序一致（脆弱）。
main_adata.obs.loc[_common_idx, "cell_type_final_subset_v1"] = (
    _sub_labels.loc[_common_idx]
)

# 处理缺失（子集中有但主 adata 中没有的细胞——不应发生，但防御处理）
_missing = set(adata_sub.obs_names) - set(_common_idx)
if _missing:
    print(f"⚠ 警告: {len(_missing)} 个子集细胞不在主 adata 中，无法回流")

_n_reflowed = main_adata.obs["cell_type_final_subset_v1"].notna().sum()
print(f"已回流: {_n_reflowed:,} 细胞 -> obs['cell_type_final_subset_v1']")
print(f"  精细标签种类: {main_adata.obs['cell_type_final_subset_v1'].nunique()}")

# 确认 cell_type_final_v1 未被修改——before/after 真实验证
# 为什么用 assert？只在回流前后做逐元素比对，不依赖从未写入的外部状态。
assert (_check == main_adata.obs["cell_type_final_v1"]).all(), (
    "cell_type_final_v1 在回流过程中被意外修改！"
)
print("  cell_type_final_v1 保持不变（before/after 真实验证通过）")

# 记录回流元数据
main_adata.uns["cell_type_final_subset_v1_notes"] = {
    "source": "stage6_5_subset",
    "subset_filter": SUBSET_FILTER,
    "subset_h5ad": OUTPUT_PATH,
    "n_cells_refined": int(_n_reflowed),
    "refined_cell_types": sorted(
        main_adata.obs["cell_type_final_subset_v1"].dropna().unique().tolist()
    ),
    "note": (
        "cell_type_final_v1 retains broad labels (all cells); "
        "cell_type_final_subset_v1 provides fine labels (subset cells only, NaN elsewhere). "
        "Downstream analyses choose the appropriate column."
    ),
    "timestamp": datetime.datetime.now().isoformat(),
}

## 写出产物

两个 h5ad 文件：
1. **子集 h5ad**（`OUTPUT_PATH`）：子集细胞的完整分析结果，可独立用于子集下游分析
2. **主 adata 更新版**（`MAIN_OUTPUT_PATH`）：含 `cell_type_final_subset_v1` 列的主图谱新版本

两个文件命名均遵循版本化约定。

In [ ]:
# 写出两个 h5ad 产物。
print("=== 写出产物 ===")

# 1. 子集 h5ad
# 内存纪律：确认 sparse + dtype
assert sp.issparse(adata_sub.X) and adata_sub.X.dtype == np.float32, (
    f"adata_sub.X 不变量被破坏: sparse={sp.issparse(adata_sub.X)}, dtype={adata_sub.X.dtype}"
)

# 统一追踪字段——子集对象的独立追踪层（与 stage4-5-6-7 命名一致）
adata_sub.uns["stage"] = "stage6_5_subset"     # 本 stage 标识
adata_sub.uns["version"] = "v1"                 # 与 OUTPUT_PATH 版本号一致
adata_sub.uns["upstream"] = [UPSTREAM_PATH]     # list 形式，支持多上游合并
adata_sub.uns["status"] = "experimental"        # PI changes to "promoted" after review

# 子集细节记录（嵌套 dict——细节层，与顶层追踪字段并存）
adata_sub.uns["stage6_5_v1"] = {
    "upstream": UPSTREAM_PATH,
    "subset_filter": SUBSET_FILTER,
    "stages_rerun": ["stage3_hvg", "stage4_embedding", "stage5_clustering", "stage6_annotation"],
    "leiden_col_used": LEIDEN_COL,
    "timestamp": datetime.datetime.now().isoformat(),
}

adata_sub.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"子集 h5ad 已写出: {OUTPUT_PATH}")
print(f"  大小: {os.path.getsize(OUTPUT_PATH):,} bytes")

# 2. 主 adata 更新版（含 cell_type_final_subset_v1）
# main_adata 是 stage 6 的产物，本 notebook 仅做回流更新（添加 cell_type_final_subset_v1 列）。
# 不放顶层 stage/version/upstream 字段——那会篡改 stage 6 的溯源链。
# 主对象的追踪字段由 stage6_annotated 负责维护，此处仅记录本次回流操作的嵌套 dict。
main_adata.uns["stage6_5_reflow_v1"] = {
    "upstream": UPSTREAM_PATH,
    "subset_h5ad": OUTPUT_PATH,
    "reflow_column": "cell_type_final_subset_v1",
    "note": "This version adds cell_type_final_subset_v1 from stage 6.5 subset re-analysis.",
    "timestamp": datetime.datetime.now().isoformat(),
}

main_adata.write_h5ad(MAIN_OUTPUT_PATH, compression="lzf")
print(f"\n主 adata 更新版已写出: {MAIN_OUTPUT_PATH}")
print(f"  大小: {os.path.getsize(MAIN_OUTPUT_PATH):,} bytes")
print(f"  含 cell_type_final_subset_v1: {'cell_type_final_subset_v1' in main_adata.obs.columns}")
print(f"  含 cell_type_final_v1: {'cell_type_final_v1' in main_adata.obs.columns}")

In [ ]:
# 内存纪律——del + gc 释放跨越 stage 边界。
# 为什么必须释放？子集 adata 虽比主 adata 小，但连同 embedding
# + obsm 矩阵仍可达到数 GB。不及时释放会累积到下游 OOM。
del adata_sub
del main_adata
gc.collect()
print("内存已释放")